# DATASCI 207 — Online (Historical-Only) Baselines (4 columns)

This notebook creates **four leakage-safe baselines** for each `(ticker, date)` row in your dataset:

- **Average-5**: rolling mean of the *known-at-time* weekly volatility over the last 5 observations  
- **Average-20**: rolling mean over the last 20 observations  
- **OLS-5**: rolling **linear trend** (OLS) fit on last 5 observations and extrapolated **5 trading days ahead**  
- **OLS-20**: same, using last 20 observations

## Key anti-leakage idea (critical)
Your target column is a **forward 5-trading-day realized volatility** (computed from returns in the future window).  
At time **t**, you cannot know `y_t` yet. The most recent forward-vol value that is fully known at time **t** is `y_{t-5}`.

So we define:
- `y = forward_vol_5d_annual_decimel_calculated`
- `y_known_at_t = y shifted by +5 rows within each ticker`

Every baseline uses **only `y_known_at_t` and earlier**, which guarantees historical-only computation.


## Mount drive

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
# =========================
# 0) CONFIG (edit this cell)
# =========================

import numpy as np
import pandas as pd

# Input CSV (edit if needed)
DATA_PATH = "/content/drive/MyDrive/Colab Notebooks/vol_dataset_post_wrangle_021026.csv"

# Output CSV with baselines appended
OUT_PATH  = "/content/drive/MyDrive/Colab Notebooks/vol_dataset_with_4_online_baselines.csv"

# Column names in your dataset
DATE_COL   = "date"
TICKER_COL = "ticker"
Y_COL      = "forward_vol_5d_annual_decimel_calculated"

# Horizon = 5 trading days ahead label
HORIZON = 5

print("Config OK. DATA_PATH =", DATA_PATH)


Config OK. DATA_PATH = /content/drive/MyDrive/Colab Notebooks/vol_dataset_post_wrangle_021026.csv


In [3]:
# =======================================
# 1) Load + validate + sort (no mutation)
# =======================================

df0 = pd.read_csv(DATA_PATH, low_memory=False)

# Basic schema checks
required = {DATE_COL, TICKER_COL, Y_COL}
missing = required - set(df0.columns)
if missing:
    raise KeyError(f"Missing required columns: {sorted(missing)}")

df = df0.copy()
df[DATE_COL] = pd.to_datetime(df[DATE_COL])

# Sort is REQUIRED for correct shift/rolling behavior
df = df.sort_values([TICKER_COL, DATE_COL]).reset_index(drop=True)

print("Shape:", df.shape)
print("Date range:", df[DATE_COL].min().date(), "→", df[DATE_COL].max().date())
print("Tickers:", df[TICKER_COL].nunique())


Shape: (96525, 23)
Date range: 2014-12-31 → 2025-12-31
Tickers: 36


## 2) Define `y_known_at_t` (what is known at time t)

### What is **y**?
- **y(t)** is the project target: forward 5-trading-day realized volatility at date **t**
- It uses returns from **t+1 … t+5**, so **it is NOT known at time t**

### What is known at time t?
- The forward-vol value whose future window ends at t is **y(t−5)**
- So we compute:

`y_known_at_t = groupby(ticker).shift(HORIZON)`

This is the core step that makes baselines leakage-safe.


In [4]:
# =============================================
# 2) Known-at-time series: y_known_at_t = y(t-5)
# =============================================

df["y_known_at_t"] = df.groupby(TICKER_COL, sort=False)[Y_COL].shift(HORIZON)

# Show quick preview for one ticker
one_ticker = df[TICKER_COL].iloc[0]
preview = df[df[TICKER_COL] == one_ticker][[DATE_COL, Y_COL, "y_known_at_t"]].head(12)
preview


,date,forward_vol_5d_annual_decimel_calculated,y_known_at_t
0,2021-04-21,0.0447,NaN
1,2021-04-22,0.0628,NaN
2,2021-04-23,0.1131,NaN
3,2021-04-26,0.1119,NaN
4,2021-04-27,0.1400,NaN
5,2021-04-28,0.1068,0.0447
6,2021-04-29,0.1227,0.0628
7,2021-04-30,0.2845,0.1131
8,2021-05-03,0.3002,0.1119
9,2021-05-04,0.2937,0.1400


## 3) Baselines: Average-5 and Average-20 (rolling means)

**Definition (for each ticker):**
- `baseline_avg_5(t)  = mean( y_known_at_t(t), …, y_known_at_t(t-4) )`
- `baseline_avg_20(t) = mean( y_known_at_t(t), …, y_known_at_t(t-19) )`

These are straightforward rolling means on the **known-at-time** series.


In [5]:
# ====================================================
# 3) Average baselines (rolling means on y_known_at_t)
# ====================================================

g = df.groupby(TICKER_COL, sort=False)["y_known_at_t"]

df["baseline_avg_5"] = (
    g.rolling(window=5, min_periods=5)
     .mean()
     .reset_index(level=0, drop=True)
)

df["baseline_avg_20"] = (
    g.rolling(window=20, min_periods=20)
     .mean()
     .reset_index(level=0, drop=True)
)

df[[DATE_COL, TICKER_COL, "baseline_avg_5", "baseline_avg_20"]].head(10)


,date,ticker,baseline_avg_5,baseline_avg_20
0,2021-04-21,BEDZ,NaN,NaN
1,2021-04-22,BEDZ,NaN,NaN
2,2021-04-23,BEDZ,NaN,NaN
3,2021-04-26,BEDZ,NaN,NaN
4,2021-04-27,BEDZ,NaN,NaN
5,2021-04-28,BEDZ,NaN,NaN
6,2021-04-29,BEDZ,NaN,NaN
7,2021-04-30,BEDZ,NaN,NaN
8,2021-05-03,BEDZ,NaN,NaN
9,2021-05-04,BEDZ,0.0945,NaN


## 4) Baselines: OLS-5 and OLS-20 (explicit setup)

We need to be explicit about the regression:

### What is **X**?
- **X is a time index** within the rolling window: `x = 0, 1, 2, ..., window-1`

### What is **y**?
- **y is the known-at-time weekly volatility** values in that window: `y = y_known_at_t`

### What do we predict?
- We want an estimate of the **forward 5-trading-day volatility at date t**.
- Recall: `y(t) = z(t+5)` where `z(t) = y_known_at_t`.
- So after fitting the trend on the last `window` known values, we **extrapolate 5 steps ahead**.

**Prediction point:**
- last x in window = `window-1`
- forecast x = `(window-1) + HORIZON`

This yields:
- `baseline_ols_5`  using `window=5`
- `baseline_ols_20` using `window=20`

We clip negative values to 0 because volatility cannot be negative.


In [6]:
# ================================================
# 4) OLS baselines: rolling linear trend + forecast
# ================================================

def rolling_trend_forecast(series: pd.Series, window: int, horizon: int) -> pd.Series:
    """
    Rolling OLS of y on a time index within each window,
    then extrapolate 'horizon' steps ahead.

    X = 0..window-1
    y = observed values of series in the window
    predict at x_pred = (window-1) + horizon
    """
    x = np.arange(window, dtype=float)
    x_mean = x.mean()
    denom = ((x - x_mean) ** 2).sum()
    x_pred = (window - 1) + horizon

    def _fit_and_predict(y_vals: np.ndarray) -> float:
        y = y_vals.astype(float)
        y_mean = y.mean()
        slope = ((x - x_mean) * (y - y_mean)).sum() / denom
        intercept = y_mean - slope * x_mean
        return intercept + slope * x_pred

    return series.rolling(window=window, min_periods=window).apply(_fit_and_predict, raw=True)


g = df.groupby(TICKER_COL, sort=False)["y_known_at_t"]

df["baseline_ols_5"] = (
    g.apply(lambda s: rolling_trend_forecast(s, window=5, horizon=HORIZON))
     .reset_index(level=0, drop=True)
     .clip(lower=0.0)
)

df["baseline_ols_20"] = (
    g.apply(lambda s: rolling_trend_forecast(s, window=20, horizon=HORIZON))
     .reset_index(level=0, drop=True)
     .clip(lower=0.0)
)

df[[DATE_COL, TICKER_COL, "baseline_ols_5", "baseline_ols_20"]].head(10)


,date,ticker,baseline_ols_5,baseline_ols_20
0,2021-04-21,BEDZ,NaN,NaN
1,2021-04-22,BEDZ,NaN,NaN
2,2021-04-23,BEDZ,NaN,NaN
3,2021-04-26,BEDZ,NaN,NaN
4,2021-04-27,BEDZ,NaN,NaN
5,2021-04-28,BEDZ,NaN,NaN
6,2021-04-29,BEDZ,NaN,NaN
7,2021-04-30,BEDZ,NaN,NaN
8,2021-05-03,BEDZ,NaN,NaN
9,2021-05-04,BEDZ,0.26229,NaN


## 5) Sanity checks (catch leakage and alignment bugs)

### What you should see
- Early rows per ticker are `NaN` because there is not enough history (expected).
- `y_known_at_t` equals `y` shifted by 5 rows within each ticker.
- OLS baselines have no negatives after clipping.
- Baseline columns should **never** use `y` directly at time t (they use `y_known_at_t`).


In [7]:
# ==================
# 5) Sanity checks
# ==================

baseline_cols = ["baseline_avg_5", "baseline_avg_20", "baseline_ols_5", "baseline_ols_20"]

print("Missing rate by baseline column:")
print(df[baseline_cols].isna().mean().sort_values())

print("\nNegative counts (should be 0):")
print((df["baseline_ols_5"] < 0).sum(), (df["baseline_ols_20"] < 0).sum())

# Spot-check one ticker alignment
tkr = df[TICKER_COL].iloc[0]
check = df[df[TICKER_COL] == tkr][[DATE_COL, Y_COL, "y_known_at_t"] + baseline_cols].head(35)
check


Missing rate by baseline column:
baseline_avg_5     0.003357
baseline_ols_5     0.003357
baseline_avg_20    0.008951
baseline_ols_20    0.008951
dtype: float64

Negative counts (should be 0):
0 0


,date,forward_vol_5d_annual_decimel_calculated,y_known_at_t,baseline_avg_5,baseline_avg_20,baseline_ols_5,baseline_ols_20
0,2021-04-21,0.0447,NaN,NaN,NaN,NaN,NaN
1,2021-04-22,0.0628,NaN,NaN,NaN,NaN,NaN
2,2021-04-23,0.1131,NaN,NaN,NaN,NaN,NaN
3,2021-04-26,0.1119,NaN,NaN,NaN,NaN,NaN
4,2021-04-27,0.1400,NaN,NaN,NaN,NaN,NaN
5,2021-04-28,0.1068,0.0447,NaN,NaN,NaN,NaN
6,2021-04-29,0.1227,0.0628,NaN,NaN,NaN,NaN
7,2021-04-30,0.2845,0.1131,NaN,NaN,NaN,NaN
8,2021-05-03,0.3002,0.1119,NaN,NaN,NaN,NaN
9,2021-05-04,0.2937,0.1400,0.09450,NaN,0.26229,NaN


## 6) Save output

This writes a new CSV with the four baselines appended.


In [8]:
# ==========
# 6) Save
# ==========

df_out = df.copy()

df_out.to_csv(OUT_PATH, index=False)
print("Wrote:", OUT_PATH)


Wrote: /content/drive/MyDrive/Colab Notebooks/vol_dataset_with_4_online_baselines.csv
